In [ ]:
import os
import shutil
import logging
import warnings
import nest_asyncio
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.llama_parse import LlamaParse
from llama_index.llms.google_genai import GoogleGenAI

# 1. Cấu hình hệ thống
nest_asyncio.apply()
logging.basicConfig(level=logging.ERROR)
for logger_name in ["llama_index", "httpx", "sentence_transformers", "huggingface_hub"]:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

os.environ["TQDM_DISABLE"] = "True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# 2. CẤU HÌNH API KEY (Điền Key của bạn vào đây)
LLAMA_CLOUD_KEY = "..."

os.environ["GOOGLE_API_KEY"] = "..." # Thay bằng Key Google AI Studio của bạn
os.environ["LLAMA_CLOUD_API_KEY"] = LLAMA_CLOUD_KEY

# 3. Cấu hình Model & Embedding
Settings.llm = GoogleGenAI(model="models/gemini-3.1-flash-lite-preview") 
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    embed_batch_size=42 # Tối ưu cho GPU của M1
)
Settings.chunk_size = 1024
Settings.chunk_overlap = 100

# --- Cấu hình đường dẫn ---
PERSIST_DIR = "./storage"
DATA_DIR = "./data"

# Bước 0: Reset storage cũ để nạp lại từ đầu (Chỉ chạy khi thêm file mới)
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
    print("🧹 Đã dọn dẹp bộ nhớ cũ.")

# Bước 1: Khởi tạo Parser chuyên dụng cho PDF
parser = LlamaParse(
    result_type="markdown", 
    num_workers=4, 
    verbose=True,
    language="en"
)

# Bước 2: Nạp dữ liệu thủ công để kiểm soát lỗi
all_docs = []

print(f"🚀 Bắt đầu quét thư mục: {DATA_DIR}")

# A. Nạp các file Word (.docx) - Xử lý Offline cực nhanh
word_reader = SimpleDirectoryReader(input_dir=DATA_DIR, required_exts=[".docx"])
word_docs = word_reader.load_data()
all_docs.extend(word_docs)
print(f"✅ Đã nạp file Word: {[d.metadata.get('file_name') for d in word_docs][:1]}...")

# B. Nạp các file PDF (.pdf) - Xử lý qua LlamaParse OCR
pdf_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith('.pdf')]
for pdf_name in pdf_files:
    pdf_path = os.path.join(DATA_DIR, pdf_name)
    print(f"📡 Đang gửi PDF lên Cloud để OCR: {pdf_name}...")
    try:
        # Gọi trực tiếp parser
        pdf_docs = parser.load_data(pdf_path)
        
        # ÉP METADATA: Gắn tên file vào từng trang để AI nhớ nguồn
        for doc in pdf_docs:
            doc.metadata["file_name"] = pdf_name
            # LlamaParse thường trả về số trang trong doc.metadata['page_number']
            # Nếu không có, ta giữ nguyên N/A hoặc dùng số thứ tự
            
        all_docs.extend(pdf_docs)
        print(f"✅ Đã nạp xong PDF: {pdf_name}")
    except Exception as e:
        print(f"❌ Lỗi khi đọc PDF {pdf_name}: {e}")
# Bước 3: Tạo Index chung cho tất cả tài liệu
if all_docs:
    index = VectorStoreIndex.from_documents(all_docs)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print(f"✨ HOÀN THÀNH! Đã lưu {len(all_docs)} mảnh dữ liệu vào storage.")
    query_engine = index.as_query_engine(similarity_top_k=5)
else:
    print("⚠️ Cảnh báo: Không có dữ liệu nào được nạp!")

/Users/nguyenhoangminh/Documents/DUT_AI/.venv_ai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6862.51it/s]


🧹 Đã dọn dẹp bộ nhớ cũ.
🚀 Bắt đầu quét thư mục: ./data
✅ Đã nạp file Word: ['Đề cương chi tiết bài giảng TT HCM 2026.docx']...
📡 Đang gửi PDF lên Cloud để OCR: Computer Networking - 3rd edition.pdf...
Started parsing the file under job_id 591e23b0-2d0f-4acb-b123-4c77db9f9dbb
✅ Đã nạp xong PDF: Computer Networking - 3rd edition.pdf
📡 Đang gửi PDF lên Cloud để OCR: Thay_Nguyen Tan Khoi.pdf...
Error while parsing the file './data/Thay_Nguyen Tan Khoi.pdf': Event loop is closed
✅ Đã nạp xong PDF: Thay_Nguyen Tan Khoi.pdf
✨ HOÀN THÀNH! Đã lưu 849 mảnh dữ liệu vào storage.


In [2]:
def ask_with_source(question):
    response = query_engine.query(question)
    
    print("\n" + "="*50)
    print(f"🤖 ĐÁP ÁN: {response}")
    print("="*50)
    
    print("\n📍 CHI TIẾT NGUỒN DỮ LIỆU:")
    
    seen_sources = set()
    for i, node in enumerate(response.source_nodes):
        meta = node.node.metadata
        
        # Thử lấy tên file từ nhiều key khác nhau
        f_name = meta.get('file_name') or meta.get('file_path') or "Tài liệu không tên"
        f_name = os.path.basename(f_name) # Chỉ lấy tên file, bỏ đường dẫn dài
        
        # Thử lấy số trang
        page = meta.get('page_number') or meta.get('page') or meta.get('page_label') or "N/A"
        
        source_id = f"{f_name}_{page}"
        if source_id not in seen_sources:
            print(f"[{i+1}] 📄 File: {f_name} | 📍 Trang: {page} | ✨ Khớp: {node.score:.2%}")
            # In thêm một đoạn ngắn để kiểm chứng
            content = node.node.get_content().replace('\n', ' ')[:150]
            print(f"    📝 Nội dung: \"{content}...\"")
            seen_sources.add(source_id)

In [3]:
# 6. Chạy thử nghiệm
# Ví dụ: Bạn có file 'ly_thuyet.pdf' và 'baitap.docx' trong thư mục data
ask_with_source("1. Phân tích bối cảnh lịch sử trong nước và quốc tế dẫn đến ra đời Tư tưởng Hồ Chí Minh?")
ask_with_source("2. Thiết bị nào cung cấp kết nối giữa các máy tính trong cùng mạng LAN ?")


🤖 ĐÁP ÁN: Dựa trên nội dung được cung cấp, quá trình hình thành tư tưởng Hồ Chí Minh gắn liền với việc tiếp thu và vận dụng sáng tạo chủ nghĩa Mác - Lênin cũng như tinh hoa văn hóa dân tộc và nhân loại, trong đó có thể phân tích bối cảnh hình thành như sau:

**1. Về bối cảnh trong nước:**
*   **Truyền thống văn hóa dân tộc:** Đây là cội nguồn sâu xa hình thành nên tư tưởng Hồ Chí Minh. Khát vọng giải phóng dân tộc đã thôi thúc Người ra đi tìm đường cứu nước. Vốn văn hóa, chính trị và thực tiễn phong phú mà Người trang bị cho bản thân trước khi đến với chủ nghĩa Mác - Lênin đã giúp Người có năng lực tư duy độc lập, tự chủ và sáng tạo.
*   **Nhu cầu thực tiễn của cách mạng:** Xuất phát từ nhu cầu giải phóng dân tộc, Hồ Chí Minh tìm đến chủ nghĩa Mác - Lênin như một kim chỉ nam cho sự nghiệp cách mạng Việt Nam. Việc vận dụng lý luận luôn được Người gắn liền với điều kiện, hoàn cảnh cụ thể của đất nước nhằm tìm ra các chủ trương, giải pháp và đối sách phù hợp.

**2. Về bối cảnh quốc tế và